In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

import xgboost as xgb


# Importing the libraries
import polars as pl
import os
import sys
import altair as alt
import vegafusion as vf
import sklearn
import time
from datetime import date, datetime, timedelta
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder 

import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV

In [ ]:
#table = pq.read_table('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Code/data/interim/Prepped_data_20241018.parquet')
df_import = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/Aggregated_data_20241106.parquet', engine='pyarrow')
# Specify the columns you want to load to reduce memory usage
#columns_to_load = ['store_nbr', 'item_nbr', 'date', 'unit_sales', 'week_number_cum', 'onpromotion', 'perishable']  # Replace with your column names
#table = pq.read_table('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Code/data/interim/Prepped_data_20241018.parquet', columns=columns_to_load)

# Convert to Pandas DataFrame if needed
#df = table.to_pandas()
df = df_import



In [ ]:
df_import.info()

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
# # Assuming your DataFrame is named df
# filtered_df = df[df['item_family'].isin(['GROCERY I', 'BREAD/BAKERY', 'SEAFOOD'])]

# # Display the filtered DataFrame
# filtered_df.info()

# df = filtered_df

In [ ]:
# # Step 1: Get the total number of unique stores
# total_stores = df['store_nbr'].nunique()

# # Step 2: Find item_nbr values present in every store_nbr
# items_in_all_stores = (
#     df.groupby('item_nbr')['store_nbr'].nunique()  # Count unique stores for each item
#     .loc[lambda x: x == total_stores]              # Keep only items that appear in all stores
#     .index                                         # Get the item numbers
# )

# # Step 3: Filter the DataFrame to keep only those item_nbr values
# filtered_df = df[df['item_nbr'].isin(items_in_all_stores)]

# # Display the filtered DataFrame
# filtered_df.info()

# df = filtered_df

In [ ]:
df['item_family'].nunique()

In [ ]:
def add_lagged_features(group, max_lag=10):
    # Ensure sorting by week_number_cum
    group = group.sort_values('week_number_cum')
    
    # Generate lagged features up to max_lag weeks back
    for lag in range(1, max_lag + 1):
        group[f'unit_sales_lag_{lag}'] = group['unit_sales'].shift(lag)
    
    return group

In [ ]:

df_lag = df.groupby(['item_nbr', 'store_nbr'], group_keys=False).apply(add_lagged_features).reset_index(drop=True)

In [ ]:
df_lag.head()

In [ ]:
df = df_lag

In [ ]:
# Assume `df` is your DataFrame and `unit_sales` is the target column
# List of features excluding the target column and any other non-numeric fields if needed
list_features = [col for col in df.columns if col not in ['unit_sales', 'store_type', 'item_family']]

# Calculate correlation matrix
correlation_matrix = df[list_features + ['unit_sales']].corr()

# Get correlations between each feature and 'unit_sales' only, drop the correlation of 'unit_sales' with itself
correlations_with_y = correlation_matrix['unit_sales'].drop('unit_sales')

# Convert to DataFrame and sort by correlation
correlations_df = correlations_with_y.sort_values(ascending=False).reset_index()
correlations_df.columns = ['Feature', 'Correlation_with_unit_sales']

# Display the resulting DataFrame
correlations_df

In [ ]:
# Define the threshold for coloring
threshold = 0.60

# Define colors: one color for correlations above the threshold, another for below
colors = ['skyblue' if corr <= threshold else 'orange' for corr in correlation_df['Correlation']]

# Plotting the bar plot with conditional coloring
plt.figure(figsize=(12, 6))
plt.bar(correlations_df['Feature'], correlations_df['Correlation_with_unit_sales'], color=colors)
plt.xlabel('Lag Feature')
plt.ylabel('Correlation with Unit Sales')
plt.title('Correlation of Unit Sales with Lagged Features')
plt.xticks(rotation=45)
plt.show()


In [ ]:
df.head()

In [ ]:
df_train = df[(df['week_number_cum'] > 138) & (df['week_number_cum'] <= 190)]
#df_train = df[df['week_number_cum'] <= 190]
df_test = df[(df['week_number_cum'] > 190) & (df['week_number_cum'] <= 216)]
#df_test = df[df['week_number_cum'] == 192]
df_validate = df[df['week_number_cum'] > 216]

In [ ]:
list_features = [col for col in df.columns if col not in ['unit_sales', 'date', 'unit_sales_lag_1']]

list_features

In [ ]:
X_train = df_train[list_features]
y_train = df_train['unit_sales']

X_test = df_test[list_features]
y_test = df_test['unit_sales']

In [ ]:
# from sklearn.metrics import make_scorer, mean_absolute_percentage_error

# # Define the model
# model = xgb.XGBRegressor(objective='reg:squarederror', enable_categorical=True)

# # Define the MAPE scoring function
# mape_scorer = make_scorer(mean_absolute_percentage_error, greater_is_better=False)

# # Set up a simplified parameter grid for tuning
# param_grid = {
#     'learning_rate': [0.05, 0.1],
#     'n_estimators': [100, 200],
#     'max_depth': [3, 5]
# }

# # Set up GridSearchCV with MAPE scoring
# grid_search = GridSearchCV(estimator=model, param_grid=param_grid, 
#                            cv=3,  # 3-fold cross-validation
#                            scoring=mape_scorer,  # Use MAPE as the scoring metric
#                            verbose=1, 
#                            n_jobs=-1)

# # Fit GridSearchCV
# grid_search.fit(X_train, y_train)

# # Get the best parameters and model
# best_params = grid_search.best_params_
# best_model = grid_search.best_estimator_

# print("Best Parameters:", best_params)



In [ ]:
model = xgb.XGBRegressor(objective='reg:squarederror', 
                              enable_categorical=True,
                              learning_rate=0.1, 
                              max_depth=5, 
                              n_estimators=200)

model.fit(X_train, y_train)


In [ ]:
feature_importance = model.feature_importances_
feature_importance

In [ ]:
y_pred = model.predict(X_test)
y_pred[y_pred < 0] = 0
y_pred

In [ ]:
df_pred = df_test.copy()
df_pred['y_pred'] = y_pred
df_pred

In [ ]:
df_pred['bias'] = df_pred['unit_sales'] - df_pred['y_pred']
df_pred

In [ ]:
# Calculate MAPE (Mean Absolute Percentage Error)
mape = (np.mean(np.abs(y_test - y_pred))) / np.mean(y_test) * 100
print(f'MAPE: {mape:.2f}%')

# Calculate Accuracy 
#accuracy = np.mean(1 - (np.abs(y_test[non_zero_indices] - y_pred[non_zero_indices]) / y_test[non_zero_indices])) * 100
accuracy = (1 - (np.mean(np.abs(y_test - y_pred)) / np.mean(y_test))) * 100
print(f'Accuracy: {accuracy:.2f}%')

# Calculate the residuals
residuals = y_test - y_pred

# Calculate Bias
#bias = np.mean(residuals)
bias = np.mean(df_pred['bias'])
print(f'Bias: {bias:.2f}')

# Calculate Standard Deviation of the residuals
# sd_bias = np.std(residuals)
sd_bias = np.std(df_pred['bias'])
print(f'Standard Deviation of Bias: {sd_bias:.2f}')

In [ ]:
# Step 1: Group by 'week_number_cum' and calculate the sum
#grouped_df = df_pred.groupby('week_number_cum')[['y_pred', 'unit_sales']].sum().reset_index()
grouped_df = df_pred[(df_pred['item_nbr'] == 103520) & (df_pred['store_nbr'] == 44)]

# Step 2: Plot the results
plt.figure(figsize=(12, 6))
plt.plot(grouped_df['week_number_cum'], grouped_df['y_pred'], marker='o', label='Predicted Sales (y_pred)', color='blue')
plt.plot(grouped_df['week_number_cum'], grouped_df['unit_sales'], marker='o', label='Actual Sales (unit_sales)', color='orange')

# Add titles and labels
plt.title('Sum of y_pred and Unit Sales Over Weeks')
plt.xlabel('Week Number Cumulative')
plt.ylabel('Sales')
plt.xticks(grouped_df['week_number_cum'])  # Set x-ticks to show all weeks
plt.legend()
plt.grid()

# Show the plot
plt.show()

In [ ]:
# Step 1: Group by 'store_nbr' and 'week_number_cum' and calculate the sum
grouped_df = df_test.groupby(['store_nbr', 'week_number_cum'])[['y_pred', 'unit_sales']].sum().reset_index()

# Step 2: Plot the results for each store
store_nbrs = grouped_df['store_nbr'].unique()  # Get unique store numbers

plt.figure(figsize=(14, 8))

# Create a plot for each store
for store in store_nbrs:
    store_data = grouped_df[grouped_df['store_nbr'] == store]
    plt.plot(store_data['week_number_cum'], store_data['y_pred'], marker='o', label=f'Store {store} - Predicted Sales (y_pred)')
    plt.plot(store_data['week_number_cum'], store_data['unit_sales'], marker='x', label=f'Store {store} - Actual Sales (unit_sales)', linestyle='--')

# Add titles and labels
plt.title('Sum of y_pred and Unit Sales Over Weeks per Store')
plt.xlabel('Week Number Cumulative')
plt.ylabel('Sales')
plt.xticks(store_data['week_number_cum'])  # Set x-ticks to show all weeks
plt.legend()
plt.grid()

# Show the plot
plt.show()